# 01 — FDA Backbone and Project Scope

## Purpose
Build a clean exploratory backbone from Drugs@FDA to understand what can be measured for the thesis project.

## Core questions
1. What approval-related variables are available?
2. What should the unit of analysis be?
3. What counts as an approval?
4. Can I create a first-pass controlled-substance indicator?
5. Do preliminary descriptive patterns suggest the project is feasible?

## Inputs
- data/raw/drugsatfda.zip

## Planned outputs
- data/processed/fda_backbone.csv
- output/tables/approvals_by_year.csv
- output/tables/controlled_substance_by_year.csv
- output/figures/approvals_by_year.png
- output/figures/controlled_substance_share_by_year.png

## Caveats
- controlled-substance screening is preliminary unless validated against DEA schedules
- submission dates may be missing, so review duration may not be directly observable

Let's start by setting out paths really quick:

In [1]:
from pathlib import Path
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parents[1]   # because notebook is in code/notebooks
DATA = ROOT / "data"
RAW = DATA / "raw"
INTERMEDIATE = DATA / "intermediate"
PROCESSED = DATA / "processed"
OUTPUT = ROOT / "output"
FIGURES = OUTPUT / "figures"
TABLES = OUTPUT / "tables"

for p in [INTERMEDIATE, PROCESSED, FIGURES, TABLES]:
    p.mkdir(parents=True, exist_ok=True)

## FDA Data Source

This project uses the **Drugs@FDA Data Files** provided by the U.S. Food and Drug Administration (FDA).

Source:
https://www.fda.gov/drugs/drug-approvals-and-databases/drugsfda-data-files [accessed Math 14, 2026]

These files provide a structured extract of the Drugs@FDA database and contain information on:

- drug applications (NDA, ANDA, BLA)
- submissions to those applications
- approval actions
- product formulations
- active ingredients
- sponsors / applicants

This dataset is preferred over the monthly approval reports because:

- it provides **structured relational tables**
- it allows **programmatic reproducibility**
- it includes **historical approvals**
- it allows linking between applications, products, and submissions.

### Scope

The Drugs@FDA database primarily covers:

- New Drug Applications (NDA)
- Abbreviated New Drug Applications (ANDA)
- Biologics License Applications (BLA)

However, biologics regulated through the **Center for Biologics Evaluation and Research (CBER)** may not appear in the same way as small-molecule drugs regulated by **CDER**.

### Download Information

Download date: `YYYY-MM-DD`

Downloaded file:

```
data/raw/drugsatfda.zip
```

The contents of this archive will be inspected and loaded in the next step.

In [5]:
zip_path = RAW / "dafdata20260313.zip"

print("Zip file exists:", zip_path.exists())
print("Location:", zip_path)

with zipfile.ZipFile(zip_path, "r") as z:
    file_list = z.namelist()

file_list

Zip file exists: True
Location: /Users/alexdelatorre/Desktop/econ580-thesis/data/raw/dafdata20260313.zip


['ActionTypes_Lookup.txt',
 'ApplicationDocs.txt',
 'Applications.txt',
 'ApplicationsDocsType_Lookup.txt',
 'Join_Submission_ActionTypes_Lookup.txt',
 'MarketingStatus.txt',
 'MarketingStatus_Lookup.txt',
 'Products.txt',
 'SubmissionClass_Lookup.txt',
 'SubmissionPropertyType.txt',
 'Submissions.txt',
 'TE.txt']

Let's extract explicitly unpack the dataset so the repo is transparent.

In [6]:
extract_dir = RAW / "dafdata20260313"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

print("Extracted to:", extract_dir)

Extracted to: /Users/alexdelatorre/Desktop/econ580-thesis/data/raw/dafdata20260313
